In [1]:
import src.database.scripts.sql as sql 

import requests
from datetime import datetime, timedelta, timezone
import numpy as np
import aiohttp
import asyncio

In [23]:
def sql_item():
    conn = sql.connect_pc()
    cursor = conn.cursor()
    query = """
        SELECT name, rarity, vendor_price FROM item_data
        WHERE type NOT IN ('Armor', 'Weapon', 'Accessory')
        AND vendor_price > 24
        ORDER BY vendor_price DESC
        """
    cursor.execute(query)
    return cursor.fetchall()


async def fetch(session, conn_limit, name, rarity, vendor_price):
    async with conn_limit:
        from_date = (datetime.now(timezone.utc) - timedelta(minutes=5)).strftime("%Y-%m-%dT%H:%M:%SZ")
        url = f"https://api.darkerdb.com/v1/market?item={name.replace('\'', "'")}&rarity={rarity}&from={from_date}&limit=50&sold=0"
        # url = f"https://api.darkerdb.com/v1/market?item={name.replace('\'', "'")}&rarity={rarity}&limit=50"
        response = await session.get(url)
        # await asyncio.sleep(.25)
        output = await response.json()
        response.release()

    price = [listing['price_per_unit'] for listing in output['body']]  
    if not price: price = [vendor_price]
    market_min = round(np.nanmin(price))
    q_under = len(list(filter(lambda x: x < vendor_price, price)))
    return (name, vendor_price - market_min, vendor_price, market_min, q_under)

async def fetch_2(session, conn_limit, name, rarity, vendor_price):
    async with conn_limit:
        from_date = (datetime.now(timezone.utc) - timedelta(minutes=5)).strftime("%Y-%m-%dT%H:%M:%SZ")
        url = f"https://api.darkerdb.com/v1/market?item={name.replace('\'', "'")}&rarity={rarity}&from={from_date}&limit=50&sold=0"
        response = await session.get(url)
        # await asyncio.sleep(.25)
        output = await response.json()
        response.release()

    pricwde = [listing['price_per_unit'] for listing in output['body']]  
    if not price: price = [vendor_price]
    market_min = round(np.nanmin(price))
    q_under = len(list(filter(lambda x: x < vendor_price, price)))
    return (name, vendor_price - market_min, vendor_price, market_min, q_under)

async def main(sem_limit):
    header = [('name', 'net', 'vendor', 'market_min', 'q_under')]
    sem = asyncio.Semaphore(sem_limit)
    async with aiohttp.ClientSession() as session:
        task = [fetch(session, sem, name, rarity, vendor_price) for name, rarity, vendor_price in sql_item()]
        rows = await asyncio.gather(*task)
    
    return header + list(rows)

In [27]:
e = await main(99)

e[1:] = sorted(e[1:], key=lambda x: x[1], reverse=True)
for x in e:
    print(f"{x[0]:<35} {x[1]:>10} {x[2]:>10} {x[3]:>10} {x[4]:>10}")

name                                       net     vendor market_min    q_under
Gold Ore                                   125        125          0          1
Gold Candelabra (Royal)                    100        400        300          1
Bug Shell                                   99        100          1          1
Spectral Hilt                               50        250        200          2
White Shark Fin                             30        200        170          1
Emerald (Royal)                             23        100         77          8
Chronicles of the Cursed Crown              20        500        480          4
Cave Trolls Precious Rock                   20        500        480          1
Ancient Scroll (Perfect)                    20        300        280          1
Emerald (Perfect)                           15         50         35          6
Diary of the Toz                            10        500        490          1
Dotted Gold Bangle (Perfect)            